In [ ]:
#Step 8 to convert cell-based model to subarea-based model

In [53]:
import pandas as pd
import numpy as np
import os

In [52]:
#file path
file_path = r"D:\git\imWEBs\IMWEBsInterface\resources\DemoDatasets\STC2021\SubArea\database"

In [54]:
def getSubareaFractionLookup(cellsubareafile, infile, outfile):
    #cell subare lookup table
    cell_subarea_csv_path = os.path.join(file_path, cellsubareafile)
    cell_subarea_df = pd.read_csv(cell_subarea_csv_path)

    #get number of cells of each subarea - c
    subarea_count = cell_subarea_df[cell_subarea_df['SubAreaId'] > 0]['SubAreaId'].value_counts()
    subarea_cell_count_df = subarea_count.to_frame(name = "SubAreaCellCount")
    subarea_cell_count_df.index.name = "SubAreaId"

    #cell riparian buffer/vegetative filter strip part file export from h5 file
    cell_part_txt_path = os.path.join(file_path,infile)
    cell_part_df = pd.read_csv(cell_part_txt_path, skip_blank_lines=True, names = ["PartId"], delim_whitespace = True)
    cell_part_df.index.name = "CellIndex"
        
    #get number of cells of each part - b
    part_count = cell_part_df[cell_part_df['PartId'] > 0]['PartId'].value_counts()
    part_cell_count_df = part_count.to_frame(name = "PartCellCount")
    part_cell_count_df.index.name = "PartId"

    # get number of unique combination of subarea and part - a
    cell_subarea_part_df = cell_subarea_df.merge(cell_part_df, on = "CellIndex", how = "left")
    subarea_part_count_df = pd.pivot_table(cell_subarea_part_df[(cell_subarea_part_df['SubAreaId'] > 0) & (cell_subarea_part_df['PartId'] > 0)], values='CellIndex', index=['SubAreaId','PartId'], aggfunc='count')
    subarea_part_count_df.reset_index(inplace=True)
    subarea_part_count_df = subarea_part_count_df.rename(columns={"CellIndex":"SubAreaPartCellCount"})

    #merge part and subarea count
    subarea_part_count_df = subarea_part_count_df.merge(part_cell_count_df, on = "PartId", how = "left")
    subarea_part_count_df = subarea_part_count_df.merge(subarea_cell_count_df, on = "SubAreaId", how = "left")

    #do the calculation
    subarea_part_count_df["FractionToBmp"] = np.round(subarea_part_count_df["SubAreaPartCellCount"] / subarea_part_count_df["PartCellCount"], decimals=4)
    subarea_part_count_df["FractionToSubarea"] = np.round(subarea_part_count_df["SubAreaPartCellCount"] / subarea_part_count_df["SubAreaCellCount"], decimals=4)

    #output
    subarea_part_count_df["LocationId"] = subarea_part_count_df["PartId"]
    riparian_buffer_lookup_csv_path = os.path.join(file_path,outfile)
    subarea_part_count_df.to_csv(riparian_buffer_lookup_csv_path, columns=['SubAreaId', "LocationId", "FractionToBmp", "FractionToSubarea"], index=False)


In [55]:
getSubareaFractionLookup("CellSubArea.csv", "buffer_parts.txt", "SubareaRiparianBufferLookup.csv")
getSubareaFractionLookup("CellSubArea.csv", "buffer_drainage.txt", "SubareaRiparianBufferDrainageLookup.csv")

In [56]:
getSubareaFractionLookup("CellSubArea.csv", "feedlot_parts.txt", "SubareaFeedlotLookup.csv")
getSubareaFractionLookup("CellSubArea.csv", "feedlot_drainage.txt", "SubareaFeedlotDrainageLookup.csv")